# Haiku Retrieval

**Project Name:** Haiku

## Purpose
- Publication-ready notebook for reproducible evaluation.

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys
from pathlib import Path

# Notebook lives at <repo>/downstream/ — HAIKU_ROOT points at the repo root.
HAIKU_ROOT = Path.cwd().parent
if str(HAIKU_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(HAIKU_ROOT / 'src'))

# -------- Fill these in to point at your local copies --------
EMBEDDINGS_DIR     = Path('<PATH_TO_PRECOMPUTED_EMBEDDINGS>')
METADATA_DIR       = Path('<PATH_TO_REGION_METADATA>')
BIOMARKER_LIST     = Path('<PATH_TO_BIOMARKER_LIST_PKL>')
ESM_EMBEDDINGS_DIR = Path('<PATH_TO_ESM_EMBEDDINGS>')
CODEX_DATA_DIR     = Path('<PATH_TO_CODEX_INDIVIDUAL_SAMPLES>')
SAMPLES_JSON       = HAIKU_ROOT / 'overlap_samples_final.json'
REGION_ID_MAPPING  = HAIKU_ROOT / 'resources' / 'region_id_mapping.json'
TEST_REGIONS_TXT   = Path('<PATH_TO_test_regions.txt>')
OUTPUT_DIR         = HAIKU_ROOT / 'downstream' / 'figs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from haiku.notebook_utils import setup_notebook, seed_everything
setup_notebook(project_root=str(HAIKU_ROOT))
seed_everything(42)


In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import pickle
from os.path import join
import pandas as pd
import numpy as np
import torch.nn.functional as F
import random
import json

import sys





from models import Haiku
from data import custom_collate_fn_trimodal, TrimodalDatasetViT
from utils import PerChannelSelfStandardization, CustomGaussianBlurTorch
from datetime import timedelta, datetime
import csv

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load yaml config
cfg = OmegaConf.load(HAIKU_ROOT / 'src' / 'configs' / 'config.yaml')


In [ ]:
import json
import pandas as pd


sample_dict = json.load(open(SAMPLES_JSON))
sample_ids = list(sample_dict.keys())


with open(TEST_REGIONS_TXT, 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]

sample_ids = list(set(test_ids) & set(sample_ids))

ref_ids = sorted(sample_ids)


In [ ]:

import os
from tqdm import tqdm

region_metadata_dir = str(METADATA_DIR)

# First, read all region_metadata CSVs into a dict: {region_id: df}
region_metadata = {}
metadata_files = [fname for fname in os.listdir(region_metadata_dir) if fname.endswith('.metadata.csv')]
print(f"Reading {len(metadata_files)} region metadata CSVs...")
for fname in tqdm(metadata_files, desc="Reading region metadata", total=len(metadata_files)):
    region_id = fname.split('.')[0]
    try:
        df = pd.read_csv(os.path.join(region_metadata_dir, fname))
        region_metadata[region_id] = df
    except Exception as e:
        print(f"Error reading {fname}: {e}")


In [ ]:
import torch

he_embedding = torch.load(EMBEDDINGS_DIR / 'he_embedding.pt')
codex_embedding = torch.load(EMBEDDINGS_DIR / 'codex_embedding.pt')
region_label = torch.load(EMBEDDINGS_DIR / 'region_label.pt')
virtual_codex_embedding = torch.load(EMBEDDINGS_DIR / 'virtual_codex_embedding.pt')
text_embedding = torch.load(EMBEDDINGS_DIR / 'text_embedding.pt')
musk_he_embedding = torch.load(EMBEDDINGS_DIR / 'baseline_musk_he_embedding.pt')
musk_codex_embedding = torch.load(EMBEDDINGS_DIR / 'baseline_musk_codex_embedding.pt')
musk_text_embedding = torch.load(EMBEDDINGS_DIR / 'baseline_musk_text_embedding.pt')


In [ ]:
ref_ids = sorted(sample_ids)

metadata_dict_values = {}
metadata_dict_ids = {}

keys = ['tissue_type', 'diagnosis', 'disease', 'Pathology diagnosis', 'stage', 'survival', 'type', 'survival_status']

for key in keys:
    metadata_dict_values[key] = []
    metadata_dict_ids[key] = []

print(f"Processing {len(region_label)} samples for metadata lookup...")
for i, sample in tqdm(enumerate(region_label), total=len(region_label), desc="Processing metadata"):
    #patch_id = sample['patch_id']
    #print(sample)
    region_id = ref_ids[sample].split('_')[0]
    df = region_metadata.get(region_id, None)
    if df is not None:
        for key in keys:
            if key in df['FEATURE_NAME'].values:
                if (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'nan') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'Unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == np.nan):
                    continue
                else:
                    metadata_dict_values[key].append(df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0])
                    metadata_dict_ids[key].append(i)

In [ ]:
import torch
from tqdm import tqdm
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

def compute_metrics_minibatch(
    query_embeddings,
    gallery_embeddings,
    query_ids,
    gallery_ids,
    query_labels=None,
    gallery_labels=None,
    top_ks=(1, 5, 10),
    mode='label',  # 'exact' (pair id) or 'label' (class/category)
    batch_size=256,
    device='cuda',
    return_index=False,
    return_top_k = 5,
):
    """
    Compute cross-modal retrieval metrics in minibatches with vectorized ops.

    Added metrics (macro-averaged across queries):
      - Precision@K, Recall@K, F1@K
      - nDCG@K
      - mAP@K (truncated AP)
      - Mean Rank (MR) and Median Rank (MedR) of the first relevant item
      - CMC@K (a.k.a. Hit@K; also kept as topK_acc for backward compatibility)

    Notes:
      - For queries with zero relevant items in the gallery:
          * Recall@K is defined as 0 (no relevant to retrieve).
          * AP/mAP and nDCG are defined as 0.
          * They are excluded from MR/MedR (rank stats require at least one relevant).
    """
    assert mode in ['exact', 'label']
    assert len(query_ids) == query_embeddings.size(0)
    assert len(gallery_ids) == gallery_embeddings.size(0)

    N_query = len(query_ids)
    N_gallery = len(gallery_ids)
    top_ks = tuple(sorted(set(top_ks)))

    # Build match matrix (N_query, N_gallery) with True where relevant
    print(f"\n[INFO] Building match matrix using vectorized comparison...")
    if mode == 'exact':
        match_matrix = (np.array(query_ids)[:, None] == np.array(gallery_ids)[None, :])
    else:
        if query_labels is None or gallery_labels is None:
            raise ValueError("mode='label' requires query_labels and gallery_labels.")
        match_matrix = (np.array(query_labels)[:, None] == np.array(gallery_labels)[None, :])
    match_matrix = torch.from_numpy(match_matrix).to(torch.bool)  # (N_query, N_gallery)
    print(f"[INFO] Match matrix shape: {match_matrix.shape}")

    # Normalize & move to device
    query_embeddings = F.normalize(query_embeddings, dim=1).to(device)
    gallery_embeddings = F.normalize(gallery_embeddings, dim=1).to(device)
    match_matrix = match_matrix.to(device)

    # Accumulators
    topk_hits = {k: 0 for k in top_ks}             # for Hit@K / CMC@K
    prec_at_k_sum = {k: 0.0 for k in top_ks}       # sum over queries of precision@k
    rec_at_k_sum  = {k: 0.0 for k in top_ks}       # sum over queries of recall@k
    f1_at_k_sum   = {k: 0.0 for k in top_ks}       # sum over queries of f1@k
    ndcg_at_k_sum = {k: 0.0 for k in top_ks}       # sum over queries of nDCG@k
    map_at_k_sum  = {k: 0.0 for k in top_ks}       # sum over queries of AP@k

    all_rr = []        # per-query reciprocal rank (for MRR)
    all_ap = []        # per-query AP (full length)
    ranks_first = []   # per-query rank of first relevant (1-based), only for queries with >=1 relevant

    print(f"\n[INFO] Starting evaluation with batch size {batch_size} on {device}")
    num_batches = (N_query + batch_size - 1) // batch_size

    if return_index:
        all_index = []

    with tqdm(total=num_batches, desc="Evaluating batches") as pbar:
        for start in range(0, N_query, batch_size):
            end = min(start + batch_size, N_query)
            B = end - start

            batch_query = query_embeddings[start:end]           # (B, D)
            batch_match = match_matrix[start:end].squeeze(-1)               # (B, N_gallery)

            # Cosine similarity and ranking
            sim = torch.matmul(batch_query, gallery_embeddings.T)
            #print(sim.min())# (B, N_gallery)
            sorted_idx = torch.argsort(sim, dim=1, descending=True)
            if return_index:
                all_index.append(sorted_idx[:, :return_top_k])
            sorted_match = torch.gather(batch_match, dim=1, index=sorted_idx).float()  # (B, N_gallery)

            # Any relevant?
            has_match = sorted_match.bool().any(dim=1)  # (B,)

            # ---- Hit@K / CMC@K and Top-K metrics ----
            # Precompute cumulative sums and ranks
            cumsum_rel = torch.cumsum(sorted_match, dim=1)  # (B, N_gallery)
            ranks = torch.arange(1, N_gallery + 1, device=device).float().unsqueeze(0)  # (1, N_gallery)

            # First relevant rank (1-based); define placeholder for no-match
            # Use argmax on boolean to get first True; for rows with no True this gives 0, so mask.
            first_hit_pos = sorted_match.bool().float().argmax(dim=1) + 1  # (B,)
            # Correct first_hit_pos for rows with no relevant (set to 0 so they don't affect MR/MedR)
            first_hit_pos = torch.where(has_match, first_hit_pos, torch.zeros_like(first_hit_pos))

            # Collect rank stats for queries with >=1 relevant
            if has_match.any():
                ranks_first.append(first_hit_pos[has_match])

            # MRR (0 for queries with no relevant)
            mrr = torch.where(has_match, 1.0 / first_hit_pos.clamp(min=1).float(), torch.zeros_like(first_hit_pos, dtype=torch.float))
            all_rr.append(mrr)

            # Full mAP (untruncated)
            precision_at_i = cumsum_rel / ranks  # (B, N_gallery)
            precision_at_relevant = precision_at_i * sorted_match  # zero where not relevant
            num_rel = sorted_match.sum(dim=1)  # (B,)
            ap_full = torch.where(num_rel > 0, precision_at_relevant.sum(dim=1) / num_rel, torch.zeros_like(num_rel))
            all_ap.append(ap_full)

            # For each requested K: compute Hit@K, P@K, R@K, F1@K, nDCG@K, mAP@K
            for k in top_ks:
                topk_rel = sorted_match[:, :k]                  # (B, k)
                hits = topk_rel.bool().any(dim=1).sum().item()
                topk_hits[k] += hits

                # Precision@K
                prec_k = topk_rel.sum(dim=1) / float(k)         # (B,)
                # Recall@K (denominator = #relevant per query)
                rec_k = torch.where(num_rel > 0, topk_rel.sum(dim=1) / num_rel, torch.zeros_like(num_rel))

                # F1@K (per-query), 0 if both zeros
                denom = (prec_k + rec_k).clamp(min=1e-12)
                f1_k = 2 * (prec_k * rec_k) / denom

                # nDCG@K (binary gains)
                # DCG = sum_{i=1..k} rel_i / log2(i+1)
                positions = torch.arange(1, k + 1, device=device).float().unsqueeze(0)  # (1, k)
                discounts = torch.log2(positions + 1.0)
                dcg = (topk_rel / discounts).sum(dim=1)  # (B,)
                # Ideal DCG: top min(k, num_rel) ones
                ideal_k = torch.minimum(num_rel, torch.tensor(float(k), device=device))
                # Build IDCG by summing 1/log2(i+1) for i=1..ideal_k
                # To avoid per-row loops, precompute a prefix sum of 1/log2(i+1)
                inv_log = 1.0 / torch.log2(torch.arange(2, k + 2, device=device).float())  # length k
                # Gather IDCG for each row: sum of first ideal_k terms
                # Create inclusive prefix sum
                inv_log_cumsum = torch.cumsum(inv_log, dim=0)  # (k,)
                # For ideal_k == 0 -> IDCG = 0
                idcg = torch.where(
                    ideal_k > 0,
                    inv_log_cumsum[(ideal_k.long() - 1).clamp(min=0)],
                    torch.zeros_like(ideal_k)
                )
                ndcg_k = torch.where(idcg > 0, dcg / idcg, torch.zeros_like(dcg))

                # AP@K (truncate AP at K)
                precision_at_i_k = precision_at_i[:, :k]             # (B, k)
                rel_k = sorted_match[:, :k]                          # (B, k)
                precision_at_relevant_k = precision_at_i_k * rel_k
                # denominator remains total #relevant (standard AP definition), or min(num_rel, k)? We keep standard:
                ap_k = torch.where(num_rel > 0,
                                   precision_at_relevant_k.sum(dim=1) / num_rel,
                                   torch.zeros_like(num_rel))

                # Accumulate macro sums
                prec_at_k_sum[k] += prec_k.sum().item()
                rec_at_k_sum[k]  += rec_k.sum().item()
                f1_at_k_sum[k]   += f1_k.sum().item()
                ndcg_at_k_sum[k] += ndcg_k.sum().item()
                map_at_k_sum[k]  += ap_k.sum().item()

            pbar.set_postfix({
                **{f"Hit@{k}": f"{topk_hits[k]/end:.3f}" for k in top_ks},
                "Batch mAP": f"{ap_full.mean().item():.4f}",
                "Batch MRR": f"{mrr.mean().item():.4f}"
            })
            pbar.update(1)

    # Aggregate
    num_queries = float(N_query)
    metrics = {}

    # Backward-compatible top-K accuracy (Hit@K / CMC@K)
    for k in top_ks:
        hit_rate = topk_hits[k] / num_queries
        metrics[f"top{k}_acc"] = hit_rate
        metrics[f"CMC@{k}"] = hit_rate  # alias for clarity

    # Macro-averaged P/R/F1@n, nDCG@n, mAP@n
    for k in top_ks:
        metrics[f"P@{k}"]    = prec_at_k_sum[k] / num_queries
        metrics[f"R@{k}"]    = rec_at_k_sum[k]  / num_queries
        metrics[f"F1@{k}"]   = f1_at_k_sum[k]   / num_queries
        metrics[f"nDCG@{k}"] = ndcg_at_k_sum[k] / num_queries
        metrics[f"mAP@{k}"]  = map_at_k_sum[k]  / num_queries

    # Rank-based metrics (only queries with at least one relevant)
    if len(ranks_first) > 0:
        ranks_first_all = torch.cat(ranks_first)  # 1-based
        metrics["MR"] = ranks_first_all.float().mean().item()
        metrics["MedR"] = ranks_first_all.median().item()
    else:
        metrics["MR"] = float('nan')
        metrics["MedR"] = float('nan')

    # Global metrics
    metrics["MRR"] = torch.cat(all_rr).mean().item()
    metrics["mAP"] = torch.cat(all_ap).mean().item()

    print("\n=== Final Minibatch Retrieval Metrics ===")
    for k in top_ks:
        print(f"Hit@{k} (Top-{k} Accuracy / CMC@{k}): {metrics[f'top{k}_acc']:.4f}")
        print(f"P@{k}: {metrics[f'P@{k}']:.4f} | R@{k}: {metrics[f'R@{k}']:.4f} | F1@{k}: {metrics[f'F1@{k}']:.4f} | nDCG@{k}: {metrics[f'nDCG@{k}']:.4f} | mAP@{k}: {metrics[f'mAP@{k}']:.4f}")
    print(f"Mean Rank (MR): {metrics['MR']:.4f}")
    print(f"Median Rank (MedR): {metrics['MedR']:.4f}")
    print(f"Mean Reciprocal Rank (MRR): {metrics['MRR']:.4f}")
    print(f"Mean Average Precision (mAP): {metrics['mAP']:.4f}")

    if return_index:
        return metrics, all_index
    else:
        return metrics



import numpy as np

def compute_metrics_minibatch_random_once(
    query_embeddings,
    gallery_embeddings,
    query_ids,
    gallery_ids,
    query_labels=None,
    gallery_labels=None,
    top_ks=(1, 5, 10),
    mode='label',
    batch_size=256,
    device='cuda',
    seed=42,
):
    """
    Run one random-match baseline evaluation with a fixed seed.
    - If mode='label': randomly permutes gallery_labels.
    - If mode='exact': randomly permutes gallery_ids.
    - Embeddings remain unchanged.
    """
    assert mode in ['exact', 'label']
    rng = np.random.default_rng(seed)

    if mode == 'label':
        if query_labels is None or gallery_labels is None:
            raise ValueError("mode='label' requires query_labels and gallery_labels.")
        perm = rng.permutation(len(gallery_labels))
        shuffled_gallery_labels = [gallery_labels[i] for i in perm]
        return compute_metrics_minibatch(
            query_embeddings=query_embeddings,
            gallery_embeddings=gallery_embeddings,
            query_ids=query_ids,
            gallery_ids=gallery_ids,
            query_labels=query_labels,
            gallery_labels=shuffled_gallery_labels,
            top_ks=top_ks,
            mode=mode,
            batch_size=batch_size,
            device=device
        )
    else:  # mode == 'exact'
        perm = rng.permutation(len(gallery_ids))
        shuffled_gallery_ids = [gallery_ids[i] for i in perm]
        return compute_metrics_minibatch(
            query_embeddings=query_embeddings,
            gallery_embeddings=gallery_embeddings,
            query_ids=query_ids,
            gallery_ids=shuffled_gallery_ids,
            query_labels=None,
            gallery_labels=None,
            top_ks=top_ks,
            mode=mode,
            batch_size=batch_size,
            device=device
        )

def plot_retrieval_metrics_barplot(ours_metrics, baseline_metrics, top_ks=(1, 5, 10, 20, 50), main_metrics=None, ours_label="Ours", baseline_label="Random Baseline"):
    """
    Draws a grouped barplot comparing main retrieval metrics between our method and the baseline.
    Each subplot is a metric, x-axis is top-k, y is value, two bars per k (ours/baseline).
    Args:
        ours_metrics: dict from compute_metrics_minibatch
        baseline_metrics: dict from compute_random_baseline_metrics
        top_ks: tuple/list of top-k values to plot
        main_metrics: list of metric names to plot (default: ["topK_acc", "P", "R", "F1", "nDCG", "mAP"])
        ours_label: label for our method
        baseline_label: label for baseline
    """
    import matplotlib.pyplot as plt
    import numpy as np

    if main_metrics is None:
        main_metrics = ["topK_acc", "P", "R", "F1", "nDCG", "mAP"]

    # Prepare data for each metric and top-k
    n_metrics = len(main_metrics)
    n_ks = len(top_ks)
    x = np.arange(n_ks)  # the label locations

    width = 0.35  # width of the bars

    fig, axes = plt.subplots(1, n_metrics, figsize=(4*n_metrics, 5), sharey=False)
    if n_metrics == 1:
        axes = [axes]

    for i, metric in enumerate(main_metrics):
        ours_vals = []
        baseline_vals = []
        for k in top_ks:
            ours_key = f"{metric.replace('topK_acc', 'top'+str(k)+'_acc').replace('K', str(k))}" if metric == "topK_acc" else f"{metric}@{k}"
            baseline_key = ours_key
            # For mAP@k, sometimes key is mAP@k, for topK_acc it's top{k}_acc
            ours_val = ours_metrics.get(ours_key, np.nan)
            baseline_val = baseline_metrics.get(baseline_key, np.nan)
            ours_vals.append(ours_val)
            baseline_vals.append(baseline_val)
        ax = axes[i]
        ax.bar(x - width/2, ours_vals, width, label=ours_label, color="#1f77b4")
        ax.bar(x + width/2, baseline_vals, width, label=baseline_label, color="#ff7f0e")
        ax.set_xticks(x)
        ax.set_xticklabels([str(k) for k in top_ks])
        ax.set_xlabel("Top-K")
        ax.set_title(metric.replace("topK_acc", "Top-K Accuracy").replace("P", "Precision").replace("R", "Recall").replace("F1", "F1-score").replace("nDCG", "nDCG").replace("mAP", "mAP"))
        ax.set_ylim(0, 1)
        if i == 0:
            ax.set_ylabel("Score")
        ax.legend()
        for idx, v in enumerate(ours_vals):
            ax.text(idx - width/2, v + 0.01, f"{v:.5f}", ha='center', va='bottom', fontsize=9)
        for idx, v in enumerate(baseline_vals):
            ax.text(idx + width/2, v + 0.01, f"{v:.5f}", ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
for key in metadata_dict_values:
    filtered_values = []
    filtered_ids = []
    for val, idx in zip(metadata_dict_values[key], metadata_dict_ids[key]):
        if (
            val is not None
            and str(val).lower() != 'nan'
            and str(val).lower() != 'unknown'
            and not (isinstance(val, float) and np.isnan(val))
        ):
            filtered_values.append(val)
            filtered_ids.append(idx)
    metadata_dict_values[key] = filtered_values
    metadata_dict_ids[key] = filtered_ids


In [ ]:
res_retrieval_results = {'Codex-to-HE': {}, 'HE-to-Codex': {}, 'Text-to-Codex': {}}

In [ ]:
res_1 = compute_metrics_minibatch(codex_embedding, he_embedding, torch.arange(codex_embedding.shape[0]), torch.arange(he_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cuda',mode='exact')
#res_2 = compute_metrics_minibatch_random_once(codex_embedding, he_embedding, torch.arange(codex_embedding.shape[0]), torch.arange(he_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cpu',mode='exact')
#res_2 = compute_metrics_minibatch_random_once(musk_codex_embedding, musk_embedding, torch.arange(musk_codex_embedding.shape[0]), torch.arange(musk_text_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cpu',mode='exact')
res_2 = compute_metrics_minibatch(musk_he_embedding, musk_codex_embedding, torch.arange(musk_codex_embedding.shape[0]), torch.arange(musk_he_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cuda',mode='exact')
res_retrieval_results['Codex-to-HE']['Ours'] = res_1
res_retrieval_results['Codex-to-HE']['Musk'] = res_2
plot_retrieval_metrics_barplot(res_1, res_2, top_ks=(1, 5, 10, 20, 50), main_metrics=None, ours_label="Ours", baseline_label="Random Baseline")

res_1 = compute_metrics_minibatch(he_embedding, codex_embedding, torch.arange(he_embedding.shape[0]), torch.arange(codex_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cuda',mode='exact')
#res_2 = compute_metrics_minibatch_random_once(codex_embedding, he_embedding, torch.arange(codex_embedding.shape[0]), torch.arange(he_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cpu',mode='exact')
#res_2 = compute_metrics_minibatch_random_once(musk_codex_embedding, musk_embedding, torch.arange(musk_codex_embedding.shape[0]), torch.arange(musk_text_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cpu',mode='exact')
res_2 = compute_metrics_minibatch(musk_he_embedding, musk_codex_embedding, torch.arange(musk_codex_embedding.shape[0]), torch.arange(musk_he_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cuda',mode='exact')
res_retrieval_results['HE-to-Codex']['Ours'] = res_1
res_retrieval_results['HE-to-Codex']['Musk'] = res_2

#plot_retrieval_metrics_barplot(res_1, res_2, top_ks=(1, 5, 10, 20, 50), main_metrics=None, ours_label="Ours", baseline_label="Random Baseline")


res_1 = compute_metrics_minibatch(text_embedding, codex_embedding, torch.arange(text_embedding.shape[0]), torch.arange(codex_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cuda',mode='exact')
#res_2 = compute_metrics_minibatch_random_once(codex_embedding, he_embedding, torch.arange(codex_embedding.shape[0]), torch.arange(he_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cpu',mode='exact')
#res_2 = compute_metrics_minibatch_random_once(musk_codex_embedding, musk_embedding, torch.arange(musk_codex_embedding.shape[0]), torch.arange(musk_text_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cpu',mode='exact')
res_2 = compute_metrics_minibatch(musk_text_embedding, musk_codex_embedding, torch.arange(musk_codex_embedding.shape[0]), torch.arange(musk_he_embedding.shape[0]), query_labels=region_label, gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), device='cuda',mode='exact')
res_retrieval_results['Text-to-Codex']['Ours'] = res_1
res_retrieval_results['Text-to-Codex']['Musk'] = res_2
plot_retrieval_metrics_barplot(res_1, res_2, top_ks=(1, 5, 10, 20, 50), main_metrics=None, ours_label="Ours", baseline_label="Random Baseline")


#plot_retrieval_metrics_barplot(res_1, res_2, top_ks=(1, 5, 10, 20, 50), main_metrics=None, ours_label="Ours", baseline_label="Random Baseline")

In [ ]:
import json

with open(OUTPUT_DIR / 'retrieval_results.json', 'w') as f:
    json.dump(res_retrieval_results, f)


In [ ]:
import json

with open(str(OUTPUT_DIR / 'retrieval_results.json'), 'r') as f:
    res_retrieval_results = json.load(f)

# Round every metric in res_retrieval_results to four digits (float type)
def round_metrics_4digits(metrics):
    # Handles dict at one level: metrics = {"R@1":..., ...}
    out = {}
    for k, v in metrics.items():
        try:
            num = float(v)
            out[k] = round(num, 4)
        except Exception:
            out[k] = v
    return out

# Recursively round all numbers in res_retrieval_results to four digits for all retrieval tasks/methods
for task in res_retrieval_results:
    for method in res_retrieval_results[task]:
        res_retrieval_results[task][method] = round_metrics_4digits(res_retrieval_results[task][method])

import numpy as np
import matplotlib.pyplot as plt

# Set font to Arial, large bold fonts for clarity, and ensure Illustrator compatibility/editability
plt.rcParams['font.family'] = 'Arial'    # use Arial font       # set large font size
plt.rcParams['svg.fonttype'] = 'none'    # don't convert text to path
plt.rcParams['pdf.fonttype'] = 42        # embed as TrueType for AI compatibility
plt.rcParams['ytick.labelsize'] = 10     # <<<--- Increase y-tick font size
plt.rcParams['font.weight'] = 'bold'     # make text bold
plt.rcParams['axes.labelweight'] = 'bold'# bold axis labels
plt.rcParams['axes.titleweight'] = 'bold'# bold title

from matplotlib.patches import Patch

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def plot_single_task_recall_broken_axis(
    task_name,
    methods_metrics,
    top_ks=(1, 5, 10, 20, 50),
    method_order=("Ours", "Musk"),
    break_point=0.05,
    figsize=(3.2, 4.5),
    save_path=None,
):
    """
    methods_metrics: dict like {'Ours': res_ours, 'Musk': res_musk}
    where each res_* contains keys 'R@1', 'R@5', ...
    """
    # ---- Extract values ----
    vals_per_method = {}
    for method in method_order:
        res = methods_metrics[method]
        # Ensure all values are rounded to four digits and floats for consistency
        vals_per_method[method] = np.array([round(float(res[f"R@{k}"]), 4) for k in top_ks], float)

    all_vals = np.concatenate(list(vals_per_method.values()))
    max_val = float(all_vals.max())

    # ---- Setup figure with broken axis ----
    fig, (ax_top, ax_bot) = plt.subplots(
        2,
        1,
        sharex=True,
        figsize=figsize,
        gridspec_kw={"height_ratios": [2, 1]},
    )

    x = np.arange(len(top_ks))
    n_methods = len(method_order)
    bar_width = 0.8 / n_methods

    # ---- Colors ----
    method_colors = {"Ours": "tab:blue", "Musk": "tab:orange"}

    # ---- Draw bars on BOTH axes (so tall bars fill down) ----
    for mi, method in enumerate(method_order):
        vals = vals_per_method[method]
        offset = (mi - (n_methods - 1) / 2.0) * bar_width
        color = method_colors[method]

        # Border around each bar
        border = dict(edgecolor="black", linewidth=0.6)

        # bottom panel (full bars to 0)
        ax_bot.bar(
            x + offset, vals,
            width=bar_width,
            color=color,
            **border
        )

        # top panel (same bars)
        ax_top.bar(
            x + offset, vals,
            width=bar_width,
            color=color,
            **border
        )

        # ---- Add annotations ----
        for xi, v in zip(x + offset, vals):
            if v <= break_point:
                ax = ax_bot
                ylim = break_point
                dy = 0.08 * ylim
            else:
                ax = ax_top
                ylim = max_val
                dy = 0.02 * ylim

            label = f"{v:.4f}"  # Always show four digits

            ax.text(
                xi, v + dy, label,
                ha="center", va="bottom",
                fontsize=10,
                clip_on=False
            )

    # ---- Panel limits ----
    ax_bot.set_ylim(0, break_point * 1.1)
    ax_top.set_ylim(break_point * 0.95, max_val * 1.05)

    # ---- Remove unwanted spines ----
    for ax in (ax_top, ax_bot):
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)
    ax_top.spines["bottom"].set_visible(False)

    ax_top.tick_params(labeltop=False)
    ax_bot.xaxis.tick_bottom()

    # ---- Draw diagonal break marks ----
    d = 0.008
    kwargs = dict(transform=ax_top.transAxes, color="k", clip_on=False, linewidth=0.7)
    ax_top.plot((-d, +d), (-d, +d), **kwargs)
    ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)

    kwargs.update(transform=ax_bot.transAxes)
    ax_bot.plot((-d, +d), (1 - d, 1 + d), **kwargs)
    ax_bot.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

    # ---- Labels ----
    ax_bot.set_xticks(x)
    ax_bot.set_xticklabels([f"R@{k}" for k in top_ks], rotation=0)
    ax_bot.set_xlabel("Recall@K")

    ax_top.set_ylabel("Recall")
    ax_top.set_title(task_name, fontsize=11)

    ax_top.tick_params(axis="x", bottom=False, labelbottom=False)
    ax_top.set_xticks([])

    # keep only bottom ticks on bottom axis
    #ax_bot.xaxis.tick_bottom()

    # ---- Legend ----
    legend_handles = [
        Patch(facecolor=method_colors[m], edgecolor="black", label=m)
        for m in method_order
    ]
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=len(method_order),
        fontsize=8
    )

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    # ---- Save SVG ----
    if save_path is not None:
        if not save_path.lower().endswith(".svg"):
            save_path += ".svg"
        print(f"Saved SVG: {save_path}")

    return fig, (ax_top, ax_bot)

for task in ["Codex-to-HE", "HE-to-Codex", "Text-to-Codex"]:
    fig, axes = plot_single_task_recall_broken_axis(
        task_name=task,
        methods_metrics=res_retrieval_results[task],
        top_ks=(1, 5, 10, 20, 50),
        method_order=("Ours", "Musk"),
        break_point=0.05,  # tune (e.g. 0.02) based on your values
        save_path=f"{task.replace('-', '_')}_recall.svg",
        figsize=(8, 4.5),
    )


In [ ]:
check_region = 6

res_1, index = compute_metrics_minibatch(he_embedding[(region_label==check_region).reshape(-1)], codex_embedding, torch.arange(he_embedding.shape[0])[(region_label==check_region).reshape(-1)], torch.arange(codex_embedding.shape[0]), query_labels=region_label[region_label==check_region], gallery_labels=region_label, top_ks=(1, 5, 10, 20, 50), mode='exact', device='cuda', batch_size=1024, return_index=True)

res_index =torch.cat(index, dim=0).detach().cpu().numpy()

In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch.nn.functional as F
import json
import sys


# Load yaml config
cfg = OmegaConf.load(HAIKU_ROOT / 'src' / 'configs' / 'config.yaml')

from models import Haiku
from data import custom_collate_fn_trimodal, TrimodalDatasetViT, TrimodalDatasetViTEmbedding, custom_collate_fn_embedding, TrimodalDatasetViTPickeVerion
from utils import PerChannelSelfStandardization, CustomGaussianBlurTorch

import warnings
warnings.filterwarnings("ignore")

import json
import pandas as pd


label_region_dict = json.load(open(REGION_ID_MAPPING))

sample_dict = json.load(open(SAMPLES_JSON))
sample_ids = list(sample_dict.keys())


with open(TEST_REGIONS_TXT, 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]

sample_ids = list(set(test_ids) & set(sample_ids))

ref_ids = sorted(sample_ids)

import torch

he_embedding = torch.load(EMBEDDINGS_DIR / 'he_embedding.pt')
codex_embedding = torch.load(EMBEDDINGS_DIR / 'codex_embedding.pt')
region_label = torch.load(EMBEDDINGS_DIR / 'region_label.pt')
virtual_codex_embedding = torch.load(EMBEDDINGS_DIR / 'virtual_codex_embedding.pt')
text_embedding = torch.load(EMBEDDINGS_DIR / 'text_embedding.pt')
musk_he_embedding = torch.load(EMBEDDINGS_DIR / 'musk_he_embedding.pt')

import pickle

vocab = pickle.load(open(BIOMARKER_LIST, 'rb'))

vocab[vocab == 'PGP9.5'] = 'PGP9_5'

for i in range(len(vocab)):
    if '.' in vocab[i]:
        vocab[i] = vocab[i].replace('.', '_')

cfg.model.vocab = vocab

# Load pretrained Haiku model directly from HuggingFace
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer, marker_embedding = Haiku.from_pretrained("zhihuanglab/Haiku", device=device)

from timm.data.constants import IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD

if cfg.dataset.he_transform:
    he_transform = transforms.Compose([
        transforms.Resize(384, interpolation=3, antialias=True),
        transforms.transforms.CenterCrop((384, 384)),
        transforms.Normalize(
            mean=IMAGENET_INCEPTION_MEAN,
            std=IMAGENET_INCEPTION_STD
        ),
    ])
else:
    he_transform = None

if cfg.dataset.codex_transform:
    codex_transform = [PerChannelSelfStandardization(), CustomGaussianBlurTorch(kernel_size=3, sigma=1.0)]
else:
    codex_transform = None


cfg.dataset.codex_path = str(CODEX_DATA_DIR)

sample_data = TrimodalDatasetViTPickeVerion(cfg.dataset.codex_path, cfg.dataset.he_path, cfg.dataset.text_path, ref_ids, tokenizer=tokenizer, max_len=cfg.dataset.max_length, he_transform=he_transform, codex_transform=None, return_raw=True)

inferece_dataloader = DataLoader(
    sample_data,
    batch_size=32,
    shuffle=False,
    collate_fn=custom_collate_fn_embedding,
    drop_last=False,
    num_workers=2
)


def get_region_id(batch):
    return torch.tensor([ref_ids.index(r) for r in batch['region_id']]).int().unsqueeze(1)


In [ ]:
corresponding_patches = {'codex':[], 'channels':[], 'HE':[]}

from tqdm import tqdm

for i, index in tqdm(enumerate(res_index[:10].tolist()), desc="Processing corresponding patches", total=len(res_index.tolist())):
    index_codex = []
    index_channels = []
    index_HE = []
    for k in index[:3]:
        index_codex.append(sample_data[k]['codex'])
        index_channels.append(sample_data[k]['channels'])
        index_HE.append(sample_data[k]['HandE'])
    corresponding_patches['codex'].append(index_codex)
    corresponding_patches['channels'].append(index_channels)
    corresponding_patches['HE'].append(index_HE)

In [ ]:
query_patches = {'HE':[], 'codex':[], 'channels':[]}


for i in torch.arange(codex_embedding.shape[0])[(region_label==check_region).reshape(-1)].numpy().tolist()[:10]:
    query_patches['HE'].append(sample_data[i]['HandE'])
    query_patches['codex'].append(sample_data[i]['codex'])
    query_patches['channels'].append(sample_data[i]['channels'])


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from matplotlib import cm as mpl_cm

import matplotlib as mpl

# ---------------- config ----------------
OUT_DIR = str(OUTPUT_DIR / 'retrieval_panels')
os.makedirs(OUT_DIR, exist_ok=True)

TOP_K = 15                      # unified Top-K channels (if more channels exist)
ALPHA = 0.8                     # composite blending strength
GAMMA = 1.0                     # gamma for channel shaping
DPI = 140
CMAP_NAME = "tab20"             # for per-channel color mapping
ABUNDANCE_WIDTH = 0.8           # grouped bar width
SHOW_FIG = False              # True: plt.show(); False: save to PDF/PNG
ROWS_PER_PAGE = None

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['pdf.fonttype'] = 3    # Use Type 3 fonts to make PDF text editable

# ---------------- helpers ----------------
def _to_hwc_rgb01(he):
    x = np.asarray(he)
    assert x.ndim == 3 and (x.shape[0] == 3 or x.shape[-1] == 3), f"HE must be (3,H,W) or (H,W,3); got {x.shape}"
    if x.shape[0] == 3:
        x = np.moveaxis(x, 0, -1)
    if x.dtype != np.uint8:
        mn, mx = float(x.min()), float(x.max())
        if mx > mn:
            x = (x - mn) / (mx - mn) * 255.0
        x = x.astype(np.uint8)
    pil = Image.fromarray(x).convert("RGB")
    return np.asarray(pil, dtype=np.float32) / 255.0  # (H,W,3) float[0,1]

def _resize_hwc(img_hwc01, target_hw):
    H, W = target_hw
    pil = Image.fromarray((np.clip(img_hwc01,0,1)*255).astype(np.uint8))
    pil = pil.resize((W, H), resample=Image.BILINEAR)
    return np.asarray(pil, dtype=np.float32) / 255.0

def _norm01_if_needed(x):
    x = np.asarray(x).astype(np.float32)
    if x.max() > 1.0:
        x = x / 255.0
    return x

'''def build_marker_to_color(all_channel_name_lists, cmap_name=CMAP_NAME):
    uniq = []
    seen = set()
    for names in all_channel_name_lists:
        for n in names:
            if n not in seen:
                uniq.append(n); seen.add(n)
    cmap = mpl_cm.get_cmap(cmap_name, max(40, len(uniq)))
    m2c = {uniq[i]: torch.tensor(cmap(i)[:3], dtype=torch.float32) for i in range(len(uniq))}
    return m2c'''

def build_marker_to_color(all_channel_name_lists):
    uniq = []
    seen = set()
    for names in all_channel_name_lists:
        for n in names:
            if n not in seen:
                uniq.append(n); seen.add(n)

    mats = [
        mpl_cm.get_cmap("RdBu"),
        mpl_cm.get_cmap("RdYlBu"),
        mpl_cm.get_cmap("PiYG"),
    ]

    N = len(uniq)

    colors = []
    for i in range(N):
        cmap = mats[i % len(mats)]
        colors.append(cmap(i / max(1, N-1))[:3])

    return {uniq[i]: torch.tensor(colors[i], dtype=torch.float32) for i in range(N)}

def codex_multichannel_to_rgb_tensor(codex_chw, ch_names, marker_to_color, alpha=ALPHA, gamma=GAMMA):
    """
    codex_chw: (C,H,W) np/torch in [0,1] or [0,255]
    returns: (3,H,W) torch.float in [0,1] composite using fixed per-channel colors
    """
    if not isinstance(codex_chw, torch.Tensor):
        codex = torch.from_numpy(np.asarray(codex_chw))
    else:
        codex = codex_chw
    codex = codex.float()
    if codex.max() > 1.0:
        codex = codex / 255.0
    print(codex.shape)
    C, H, W = codex.shape
    rgb = torch.zeros(3, H, W, dtype=torch.float32)
    for j, name in enumerate(ch_names):
        ch = codex[j]
        mn, mx = torch.amin(ch), torch.amax(ch)
        if (mx - mn) <= 1e-12:
            continue
        chn = (ch - mn) / (mx - mn)
        if gamma != 1.0:
            chn = chn.clamp(0,1) ** gamma
        color = marker_to_color.get(name, torch.tensor([0.5,0.5,0.5], dtype=torch.float32)).view(3,1,1)
        a = (alpha * chn).view(1, H, W)
        rgb = rgb * (1 - a) + color * a
    return rgb.clamp(0,1)

def select_overlap_and_topk(stacks_chw, name_lists, top_k=TOP_K):
    """
    stacks_chw: list of (C,H,W) arrays/tensors for versions [Q, R1..Rk]
    name_lists: list of channel-name lists aligned with stacks_chw
    Returns: selected_names (len<=top_k), version_means (V x K)
    """
    V = len(stacks_chw)
    overlap = set(name_lists[0])
    for v in range(1, V):
        overlap &= set(name_lists[v])
    if not overlap:
        return [], None

    overlap = sorted(list(overlap))
    # means per version on overlap
    means = []
    for v in range(V):
        stack = _norm01_if_needed(stacks_chw[v])
        names = name_lists[v]
        n2i = {n:i for i,n in enumerate(names)}
        mv = [float(stack[n2i[n]].mean()) for n in overlap]
        means.append(mv)
    means = np.asarray(means)  # [V, |overlap|]
    global_mean = means.mean(axis=0)
    order = np.argsort(-global_mean)
    K = min(top_k, len(overlap))
    sel_idx = order[:K]
    selected_names = [overlap[k] for k in sel_idx]
    version_means = means[:, sel_idx]  # [V,K]
    return selected_names, version_means

def composite_from_selected(stack_chw, selected_names, full_names):
    """
    Filter (C,H,W) by selected_names according to full_names order.
    Return (Csel,H,W) np.float32[0,1] and the kept names.
    """
    n2i = {n:i for i,n in enumerate(full_names)}
    idxs = [n2i[n] for n in selected_names if n in n2i]
    sel = _norm01_if_needed(np.asarray(stack_chw))[idxs]
    return sel.astype(np.float32), [full_names[i] for i in idxs]

# ---------------- drawing panels ----------------
def draw_query_retrieval_pairs_panel(pid, he_list, codex_list, chan_list, marker_to_color, selected_names, out_dir=OUT_DIR, dpi=DPI):
    """
    he_list: [HE_Q, HE_R1, HE_R2, ...] each (3,H,W) or (H,W,3)
    codex_list: [Cdx_Q, Cdx_R1, ...] each (C,H,W)
    chan_list: [Names_Q, Names_R1, ...]
    selected_names: unified Top-K channel names
    Draw N_rows × 2_cols: left=HE, right=CODEX composite; first row is Query.
    """
    V = len(he_list)
    # prepare composites (only selected_names)
    composites = []
    he_imgs = []
    for v in range(V):
        he_hwcrgb = _to_hwc_rgb01(he_list[v])
        he_imgs.append(he_hwcrgb)
        sel_stack, sel_names = composite_from_selected(codex_list[v], selected_names, chan_list[v])
        rgb_t = codex_multichannel_to_rgb_tensor(sel_stack, sel_names, marker_to_color, alpha=ALPHA, gamma=GAMMA)
        rgb = np.transpose(rgb_t.numpy(), (1,2,0))
        if rgb.shape[:2] != he_hwcrgb.shape[:2]:
            rgb = _resize_hwc(rgb, he_hwcrgb.shape[:2])
        composites.append(rgb)

    H, W = he_imgs[0].shape[:2]
    fig_w = max(5.0, (2*W)/dpi)
    fig_h = max(4.0, (V*H)/dpi)

    fig, axes = plt.subplots(nrows=V, ncols=2, figsize=(fig_w, fig_h), dpi=dpi)
    if V == 1:
        axes = np.array([axes])

    for i in range(V):
        # HE
        axes[i,0].imshow(he_imgs[i]); axes[i,0].set_axis_off()
        title_l = "Query HE" if i==0 else f"Top{i} HE"
        axes[i,0].set_title(title_l, fontsize=9, pad=1.5)
        # CODEX composite
        axes[i,1].imshow(composites[i]); axes[i,1].set_axis_off()
        title_r = "Query CODEX (composite)" if i==0 else f"Top{i} CODEX (composite)"
        axes[i,1].set_title(title_r, fontsize=9, pad=1.5)

    fig.suptitle(f"{pid} — Query + Retrieval (Left: HE | Right: CODEX composite)", fontsize=12, y=1.02)
    plt.tight_layout(pad=0.15, w_pad=0.3, h_pad=0.25)
    if SHOW_FIG:
        plt.show()
    else:
        outp = os.path.join(out_dir, f"{pid}_pairs_panel.pdf")
        plt.close(fig)
        print(f"[saved] {outp}")

def draw_abundance_panel(pid, selected_names, version_means, out_dir=OUT_DIR, dpi=DPI):
    """
    version_means: [V,K] with V = Query + K retrievals.
    Plot bars: Query first, then R1..Rk for each selected channel.
    """
    V, K = version_means.shape
    x = np.arange(K)
    width = ABUNDANCE_WIDTH / V

    fig, ax = plt.subplots(figsize=(max(10, K*0.5), 4), dpi=dpi)
    for v in range(V):
        label = "Query" if v == 0 else f"Top{v}"
        ax.bar(x + v*width, version_means[v], width, label=label)
    ax.set_xticks(x + width*(V-1)/2)
    ax.set_xticklabels(selected_names, rotation=70, ha='right', fontsize=9)
    ax.set_ylabel('Mean intensity (0–1)')
    ax.set_title(f'{pid} — Channel Abundance (Query first, then retrieval)')
    ax.legend(ncol=min(6,V))
    plt.tight_layout()
    if SHOW_FIG:
        plt.show()
    else:
        outp = os.path.join(out_dir, f"{pid}_abundance.pdf")
        plt.close(fig)
        print(f"[saved] {outp}")

def draw_per_biomarker_panels(pid, selected_names, stacks_chw, name_lists, marker_to_color, out_dir=OUT_DIR, dpi=DPI):
    """
    For each biomarker in selected_names, draw 1×V tiles:
      [Query, Top1, Top2, ...] of that single channel tinted with its fixed color.
    """
    V = len(stacks_chw)
    for biomarker in selected_names:
        tiles = []
        for v in range(V):
            stack = _norm01_if_needed(stacks_chw[v])
            names = name_lists[v]
            if biomarker in names:
                idx = names.index(biomarker)
                ch2d = stack[idx]  # (H,W)
                color = marker_to_color.get(biomarker, torch.tensor([0.5,0.5,0.5], dtype=torch.float32))
                # tint
                x = ch2d.astype(np.float32)
                mn, mx = float(x.min()), float(x.max())
                if mx > mn:
                    x = (x - mn) / (mx - mn)
                if GAMMA != 1.0:
                    x = np.power(np.clip(x,0,1), GAMMA)
                c = np.asarray(color.numpy(), dtype=np.float32)
                rgb = (x[...,None] * c[None,None,:]).clip(0,1)  # (H,W,3)
            else:
                H, W = stacks_chw[v].shape[-2], stacks_chw[v].shape[-1]
                rgb = np.zeros((H,W,3), dtype=np.float32)
            # match query HE size for consistent display if needed
            tiles.append(rgb)

        fig, axes = plt.subplots(1, V, figsize=(4*V, 4), dpi=dpi)
        if V == 1: axes = [axes]
        for v in range(V):
            axes[v].imshow(tiles[v]); axes[v].axis("off")
            axes[v].set_title("Query" if v==0 else f"Top{v}", fontsize=10)
        fig.suptitle(f"{pid} — {biomarker}", fontsize=12)
        plt.tight_layout(pad=0.5)
        if SHOW_FIG:
            plt.show()
        else:
            outp = os.path.join(out_dir, f"{pid}_biomarker_{biomarker}.svg")
            plt.close(fig)
            print(f"[saved] {outp}")

# ---------------- main driver ----------------
def render_query_and_retrieval(
    query_patches,        # {'HE': [..], 'codex':[ (C,H,W) ..], 'channels':[ [names] ..]}
    corresponding_patches,# {'HE': [ [..]*K ], 'codex':[ [ (C,H,W) ]*K ], 'channels': [ [names]*K ]}
    ids=None,             # optional names for each query
    top_k_channels=TOP_K
):
    N = len(query_patches['HE'])
    if ids is None:
        ids = [f"query_{i:03d}" for i in range(N)]

    # build global marker color map from ALL channel lists (query + all retrievals)
    all_name_lists = []
    for i in range(N):
        all_name_lists.append(query_patches['channels'][i])
        for names in corresponding_patches['channels'][i]:
            all_name_lists.append(names)
    marker_to_color = build_marker_to_color(all_name_lists)

    for i in range(N):
        pid = ids[i]
        # versions = [Query] + retrievals
        he_versions = [query_patches['HE'][i]] + corresponding_patches['HE'][i]
        codex_versions = [query_patches['codex'][i]] + corresponding_patches['codex'][i]
        names_versions = [query_patches['channels'][i]] + corresponding_patches['channels'][i]

        # choose unified Top-K on the overlap across versions (Query first, then Top1..K)
        selected_names, version_means = select_overlap_and_topk(codex_versions, names_versions, top_k=top_k_channels)
        if len(selected_names) == 0:
            print(f"[{pid}] no overlapping channels; skip.")
            continue

        # 1) HE | CODEX composite pairs panel (Query first, then retrievals)
        draw_query_retrieval_pairs_panel(pid, he_versions, codex_versions, names_versions, marker_to_color, selected_names, out_dir=OUT_DIR, dpi=DPI)

        # 2) Abundance panel (bar chart): Query first, then Top1..TopK
        #draw_abundance_panel(pid, selected_names, version_means, out_dir=OUT_DIR, dpi=DPI)

        # 3) Per-biomarker panels: a row of [Query, Top1, Top2, ...] for each selected marker
       # draw_per_biomarker_panels(pid, selected_names, codex_versions, names_versions, marker_to_color, out_dir=OUT_DIR, dpi=DPI)

        print(f"[{pid}] done. Selected {len(selected_names)} unified channels.")


render_query_and_retrieval(query_patches, corresponding_patches, top_k_channels=12)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from matplotlib import cm as mpl_cm
import matplotlib as mpl

from mpl_toolkits.axes_grid1 import ImageGrid

plt.rcParams['svg.fonttype'] = 'none'

# Set default font to Arial and increase font sizes
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10       # Default text
plt.rıcParams['axes.titlesize'] = 18   # Axes title
plt.rcParams['axes.labelsize'] = 14   # Axis label size
plt.rcParams['xtick.labelsize'] = 10  # Tick label
plt.rcParams['ytick.labelsize'] = 10  # Tick label
# ---------------- config ----------------
OUT_DIR = str(OUTPUT_DIR / 'retrieval_panel')
os.makedirs(OUT_DIR, exist_ok=True)

TOP_K = 15                      # unified Top-K channels (if more channels exist)
ALPHA = 0.8                     # composite blending strength
GAMMA = 1.0                     # gamma for channel shaping
DPI = 140
CMAP_NAME = "tab20"             # for per-channel color mapping
ABUNDANCE_WIDTH = 0.8           # grouped bar width
SHOW_FIG = False                # True: plt.show(); False: save to PDF/PNG
ROWS_PER_PAGE = None

mpl.rcParams['svg.fonttype'] = 'none'

# ---------------- helpers ----------------

def z_to_unit_np(x, zmin=-3.0, zmax=3.0):
    x = np.asarray(x, dtype=np.float32)
    x = np.clip(x, zmin, zmax)
    return (x - zmin) / (zmax - zmin + 1e-8)

def z_to_unit_torch(x, zmin=-3.0, zmax=3.0):
    x = x.float()
    x = torch.clamp(x, zmin, zmax)
    return (x - zmin) / (zmax - zmin + 1e-8)

def _to_hwc_rgb01(he):
    x = np.asarray(he)
    assert x.ndim == 3 and (x.shape[0] == 3 or x.shape[-1] == 3), f"HE must be (3,H,W) or (H,W,3); got {x.shape}"
    if x.shape[0] == 3:
        x = np.moveaxis(x, 0, -1)
    if x.dtype != np.uint8:
        mn, mx = float(x.min()), float(x.max())
        if mx > mn:
            x = (x - mn) / (mx - mn) * 255.0
        x = x.astype(np.uint8)
    pil = Image.fromarray(x).convert("RGB")
    return np.asarray(pil, dtype=np.float32) / 255.0  # (H,W,3) float[0,1]

def _resize_hwc(img_hwc01, target_hw):
    H, W = target_hw
    pil = Image.fromarray((np.clip(img_hwc01, 0, 1) * 255).astype(np.uint8))
    pil = pil.resize((W, H), resample=Image.BILINEAR)
    return np.asarray(pil, dtype=np.float32) / 255.0

def _norm01_if_needed(x):
    x = np.asarray(x).astype(np.float32)
    if x.size == 0:
        return x
    if x.min() < 0:
        return z_to_unit_np(x)
    if x.max() > 1.0:
        x = x / 255.0
    return x

def build_marker_to_color(all_channel_name_lists, cmap_name=CMAP_NAME):
    uniq = []
    seen = set()
    for names in all_channel_name_lists:
        for n in names:
            if n not in seen:
                uniq.append(n)
                seen.add(n)
    cmap = mpl_cm.get_cmap(cmap_name, max(40, len(uniq)))
    m2c = {uniq[i]: torch.tensor(cmap(i)[:3], dtype=torch.float32) for i in range(len(uniq))}
    return m2c

def codex_multichannel_to_rgb_tensor(codex_chw, ch_names, marker_to_color, alpha=ALPHA, gamma=GAMMA):
    if not isinstance(codex_chw, torch.Tensor):
        codex = torch.from_numpy(np.asarray(codex_chw))
    else:
        codex = codex_chw
    codex = codex.float()

    if codex.numel() > 0 and codex.min() < 0:
        codex = z_to_unit_torch(codex)
    elif codex.max() > 1.0:
        codex = codex / 255.0

    C, H, W = codex.shape
    rgb = torch.zeros(3, H, W, dtype=torch.float32)

    for j, name in enumerate(ch_names):
        ch = codex[j]
        if (ch.max() - ch.min()) <= 1e-6:
            continue
        chn = ch.clamp(0, 1)
        if gamma != 1.0:
            chn = chn ** gamma

        color = marker_to_color.get(
            name,
            torch.tensor([0.5, 0.5, 0.5], dtype=torch.float32)
        ).view(3, 1, 1)

        a = (alpha * chn).view(1, H, W)
        rgb = rgb * (1 - a) + color * a

    return rgb.clamp(0, 1)

def _channel_contrast_score(ch2d):
    flat = ch2d.reshape(-1)
    p1 = np.percentile(flat, 1.0)
    p99 = np.percentile(flat, 99.0)
    return float(p99 - p1)

def select_overlap_and_topk(stacks_chw, name_lists, top_k=TOP_K):
    V = len(stacks_chw)
    overlap = set(name_lists[0])
    for v in range(1, V):
        overlap &= set(name_lists[v])
    if not overlap:
        return [], None

    overlap = sorted(list(overlap))

    stacks01 = [_norm01_if_needed(s) for s in stacks_chw]

    query_stack = stacks01[0]
    q_names = name_lists[0]
    q_n2i = {n: i for i, n in enumerate(q_names)}

    contrast_scores = []
    for n in overlap:
        idx = q_n2i[n]
        ch2d = query_stack[idx]
        contrast_scores.append(_channel_contrast_score(ch2d))
    contrast_scores = np.asarray(contrast_scores)
    order = np.argsort(-contrast_scores)
    K = min(top_k, len(overlap))
    sel_idx = order[:K]
    selected_names = [overlap[k] for k in sel_idx]

    means = []
    for v in range(V):
        stack_v = stacks01[v]
        names_v = name_lists[v]
        n2i_v = {n: i for i, n in enumerate(names_v)}
        mv = [float(stack_v[n2i_v[n]].mean()) for n in selected_names]
        means.append(mv)
    version_means = np.asarray(means)

    return selected_names, version_means

def composite_from_selected(stack_chw, selected_names, full_names):
    n2i = {n: i for i, n in enumerate(full_names)}
    idxs = [n2i[n] for n in selected_names if n in n2i]
    sel = np.asarray(stack_chw, dtype=np.float32)[idxs]
    sel = _norm01_if_needed(sel)
    return sel.astype(np.float32), [full_names[i] for i in idxs]

# ---------------- drawing panels ----------------

def draw_query_retrieval_pairs_panel(pid, he_list, codex_list, chan_list, marker_to_color, selected_names, out_dir=OUT_DIR, dpi=DPI):
    V = len(he_list)
    composites = []
    he_imgs = []
    for v in range(V):
        he_hwcrgb = _to_hwc_rgb01(he_list[v])
        he_imgs.append(he_hwcrgb)
        sel_stack, sel_names = composite_from_selected(
            codex_list[v], selected_names, chan_list[v]
        )
        rgb_t = codex_multichannel_to_rgb_tensor(
            sel_stack, sel_names, marker_to_color, alpha=ALPHA, gamma=GAMMA
        )
        rgb = np.transpose(rgb_t.numpy(), (1, 2, 0))
        if rgb.shape[:2] != he_hwcrgb.shape[:2]:
            rgb = _resize_hwc(rgb, he_hwcrgb.shape[:2])
        composites.append(rgb)

    H, W = he_imgs[0].shape[:2]
    fig_w = max(5.0, (2 * W) / dpi)
    fig_h = max(4.0, (V * H) / dpi)

    fig, axes = plt.subplots(nrows=V, ncols=2, figsize=(fig_w, fig_h), dpi=dpi)
    if V == 1:
        axes = np.array([axes])

    for i in range(V):
        axes[i, 0].imshow(he_imgs[i])
        axes[i, 0].set_axis_off()
        title_l = "Query HE" if i == 0 else f"Top{i} HE"
        #axes[i, 0].set_title(title_l, fontsize=9, pad=1.5)

        axes[i, 1].imshow(composites[i])
        axes[i, 1].set_axis_off()
        title_r = "Query CODEX (composite)" if i == 0 else f"Top{i} CODEX (composite)"
        #axes[i, 1].set_title(title_r, fontsize=9, pad=1.5)

    fig.suptitle(
        f"{pid} — Query + Retrieval (Left: HE | Right: CODEX composite)",
        fontsize=12,
        y=1.02,
    )
    plt.tight_layout(pad=0.15, w_pad=0.3, h_pad=0.5)
    if SHOW_FIG:
        plt.show()
    else:
        outp = os.path.join(out_dir, f"{pid}_pairs_panel.svg")
        plt.close(fig)
        print(f"[saved] {outp}")

def draw_abundance_panel(pid, selected_names, version_means, out_dir=OUT_DIR, dpi=DPI):
    V, K = version_means.shape
    x = np.arange(K)
    width = ABUNDANCE_WIDTH / V

    fig, ax = plt.subplots(figsize=(max(10, K * 0.5), 4), dpi=dpi)
    for v in range(V):
        label = "Query" if v == 0 else f"Top{v}"
        ax.bar(x + v * width, version_means[v], width, label=label)
    ax.set_xticks(x + width * (V - 1) / 2)
    ax.set_xticklabels(selected_names, rotation=70, ha="right", fontsize=9)
    ax.set_ylabel("Mean intensity (0–1)")
    ax.set_title(f"{pid} — Channel Abundance (Query first, then retrieval)")
    ax.legend(ncol=min(6, V))
    plt.tight_layout()
    if SHOW_FIG:
        plt.show()
    else:
        outp = os.path.join(out_dir, f"{pid}_abundance.pdf")
        plt.close(fig)
        print(f"[saved] {outp}")

def draw_per_biomarker_panels(
    pid,
    selected_names,
    stacks_chw,
    name_lists,
    marker_to_color,
    out_dir=OUT_DIR,
    dpi=DPI,
    subtitle=None,
):
    """
    For each biomarker, draw one row of [Query, Top1, Top2, ...] tiles.

    Biomarker names are placed in the *gaps between rows*:
      - Label between row 0 and row 1 is selected_names[1] (CD11c in your example)
      - Label between row 1 and row 2 is selected_names[2], etc.
    All labels are:
      - black
      - centered horizontally across the grid
      - at identical relative positions between rows.

    Row spacing is controlled by a small h_pad in tight_layout, so gaps are minimal.
    """

    import matplotlib as mpl

    V = len(stacks_chw)          # versions: Query, Top1, ...
    K = len(selected_names)      # biomarkers

    if K == 0 or V == 0:
        print(f"[{pid}] no biomarkers or versions to draw.")
        return

    # --- distinct colors for tinting channels (not for text) ---
    if K <= 5:
        cmap = mpl.colormaps.get_cmap("tab10")
        marker_colors = [np.array(cmap(i % 10)[:3]) for i in range(K)]
        local_marker_to_color = {
            name: marker_colors[i] for i, name in enumerate(selected_names)
        }
    else:
        local_marker_to_color = {
            name: marker_to_color.get(
                name, np.array([0.5, 0.5, 0.5])  # fallback gray
            )
            for name in selected_names
        }

    # ---------- create figure: K rows × V columns ----------
    fig_w = max(3 * V, 6)
    row_height = 3  # slightly compact rows
    fig_h = max(row_height * K, 6)

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=dpi)

    # 2. Use ImageGrid (This fixes the sizing/aspect ratio issues)
    grid = ImageGrid(fig, 111,
                    nrows_ncols=(K, V),
                    axes_pad=0.1,  # Adjust this for space between images
                    share_all=True,
                    )

    # 3. Reshape grid to match your (K, V) structure
    axes = np.array(grid).reshape(K, V)

    # ensure axes is 2D
    if K == 1 and V == 1:
        axes = np.array([[axes]])
    elif K == 1:
        axes = axes[np.newaxis, :]
    elif V == 1:
        axes = axes[:, np.newaxis]

    # ---------- draw the biomarker rows ----------
    for bi, biomarker in enumerate(selected_names):
        color = local_marker_to_color[biomarker]
        c = np.asarray(color, dtype=np.float32)

        for v in range(V):
            ax = axes[bi, v]

            stack = np.asarray(stacks_chw[v], dtype=np.float32)
            if stack.size > 0 and stack.min() < 0:
                stack = z_to_unit_np(stack)
            else:
                stack = _norm01_if_needed(stack)

            names = name_lists[v]
            if biomarker in names:
                idx = names.index(biomarker)
                ch2d = stack[idx]
                x = ch2d
                if GAMMA != 1.0:
                    x = np.power(np.clip(x, 0, 1), GAMMA)
                rgb = (x[..., None] * c[None, None, :]).clip(0, 1)
            else:
                H, W = stacks_chw[v].shape[-2], stacks_chw[v].shape[-1]
                rgb = np.zeros((H, W, 3), dtype=np.float32)

            #print(rgb.shape)
            ax.imshow(rgb)

            ax.axis("off")

            # Put version titles only on the first biomarker row,
            # but reserve the same vertical space for all rows.
            if bi == 0:
                title_text = "Query" if v == 0 else f"Top{v}"
            else:
                title_text = " "  # blank title to keep row heights consistent

            #ax.set_title(title_text, fontsize=10, pad=2)
    # ---------- compact row spacing ----------
    # Small h_pad → small vertical gaps
    #plt.tight_layout(h_pad=2)

    # After layout, compute positions for each row
    # (tight_layout must already have been called)
    row_bounds = []
    for bi in range(K):
        row_axes = axes[bi, :]
        pos0 = row_axes[0].get_position()
        y0 = pos0.y0
        y1 = pos0.y1
        row_bounds.append((y0, y1))

    # Global horizontal center (same for all labels)
    first_row_axes = axes[0, :]
    x0 = first_row_axes[0].get_position().x0
    x1 = first_row_axes[-1].get_position().x1
    x_center = x0 + (x1 - x0) / 2.0

    # ---------- place labels in the gaps ----------
    # gap between row (i-1) and row i → label for biomarker i
    for bi in range(1, K):
        upper_y0, upper_y1 = row_bounds[bi - 1]
        lower_y0, lower_y1 = row_bounds[bi]

        # midpoint between bottom of upper row and top of lower row
        mid_y = (upper_y0 + lower_y1) / 2.0

        if subtitle is None:
            text_label = selected_names[bi]
        else:
            text_label = subtitle

        '''fig.text(
            x_center,
            mid_y,
            text_label,
            ha="center",
            va="center",
            fontsize=13,
            color="black",
            fontweight="bold",
        )'''

    if SHOW_FIG:
        plt.show()
    else:
        outp = os.path.join(out_dir, f"{pid}_biomarkers_grid.svg")
        plt.close(fig)
        print(f"[saved] {outp}")

# ---------------- main driver ----------------

def render_query_and_retrieval(
    query_patches,        # {'HE': [..], 'codex':[ (C,H,W) ..], 'channels':[ [names] ..]}
    corresponding_patches,# {'HE': [ [..]*K ], 'codex':[ [ (C,H,W) ]*K ], 'channels': [ [names]*K ]}
    ids=None,             # optional names for each query
    top_k_channels=TOP_K
):
    N = len(query_patches["HE"])
    if ids is None:
        ids = [f"query_{i:03d}" for i in range(N)]

    # build global marker color map from ALL channel lists (query + all retrievals)
    all_name_lists = []
    for i in range(N):
        all_name_lists.append(query_patches["channels"][i])
        for names in corresponding_patches["channels"][i]:
            all_name_lists.append(names)
    marker_to_color = build_marker_to_color(all_name_lists, cmap_name=CMAP_NAME)

    for i in range(N):
        pid = ids[i]
        # versions = [Query] + retrievals
        he_versions = [query_patches["HE"][i]] + corresponding_patches["HE"][i]
        codex_versions = [query_patches["codex"][i]] + corresponding_patches["codex"][i]
        names_versions = [query_patches["channels"][i]] + corresponding_patches["channels"][i]

        # choose unified Top-K on the overlap across versions (Query first, then Top1..K)
        selected_names, version_means = select_overlap_and_topk(
            codex_versions, names_versions, top_k=top_k_channels
        )
        if len(selected_names) == 0:
            print(f"[{pid}] no overlapping channels; skip.")
            continue

        # 1) HE | CODEX composite pairs panel (Query first, then retrievals)
        draw_query_retrieval_pairs_panel(
            pid,
            he_versions,
            codex_versions,
            names_versions,
            marker_to_color,
            selected_names,
            out_dir=OUT_DIR,
            dpi=DPI,
        )

        # 2) Abundance panel (bar chart): Query first, then Top1..TopK
        # if version_means is not None:
        #     draw_abundance_panel(pid, selected_names, version_means, out_dir=OUT_DIR, dpi=DPI)

        # 3) Per-biomarker panels: a row of [Query, Top1, Top2, ...] for each selected marker
        print(selected_names[:5])
        draw_per_biomarker_panels(
            pid,
            selected_names[:5],
            codex_versions,
            names_versions,
            marker_to_color,
            out_dir=OUT_DIR,
            dpi=DPI,
        )

        print(f"[{pid}] done. Selected {len(selected_names)} unified channels.")

import pickle

with open("query_patches.pkl", "rb") as f:
    query_patches = pickle.load(f)

with open("corresponding_patches.pkl", "rb") as f:
    corresponding_patches = pickle.load(f)

render_query_and_retrieval(query_patches, corresponding_patches, top_k_channels=15)

In [ ]:
import pickle

with open("query_patches.pkl", "wb") as f:

with open("corresponding_patches.pkl", "wb") as f:
